In [11]:
import pandas as pd
import numpy as np
import json
import os
import re


# 读取并清洗数据
location_df = pd.read_csv('Location.csv')
location_df.rename(columns={'Geneid':'Gene'}, inplace=True)

ncbi_gene_id_df = pd.read_csv('NCBI_gene_id.csv', names=['NCBI_gene_id', 'Gene'])
ncbi_gene_id_df['Gene'] = ncbi_gene_id_df['Gene'].str.replace('mtm:','')
ncbi_gene_id_df['NCBI_gene_id'] = ncbi_gene_id_df['NCBI_gene_id'].str.replace('ncbi-geneid:','')

protein_id_df = pd.read_csv('Protein_id.csv', names=['Protein_id', 'Gene'])
protein_id_df['Gene'] = protein_id_df['Gene'].str.replace('mtm:','')
protein_id_df['Protein_id'] = protein_id_df['Protein_id'].str.replace('ncbi-proteinid:','')

uniprot_id_df = pd.read_csv('Uniprot_id.csv', names=['Uniport_id', 'Gene'])
uniprot_id_df['Gene'] = uniprot_id_df['Gene'].str.replace('mtm:','')
uniprot_id_df['Uniport_id'] = uniprot_id_df['Uniport_id'].str.replace('up:','')

kunm_df = pd.read_csv('Knum.csv', names=['Protein_id', 'Knum'])
kunm_df['Protein_id'] = kunm_df['Protein_id'].apply(lambda x: x[:-2])
kunm_df = kunm_df.groupby("Protein_id")["Knum"].apply(lambda x: ','.join(x)).reset_index()

description_df = pd.read_csv('Description.csv', names=['Protein_id', 'Description'])

pfam_df = pd.read_csv('PFAM.csv', names=['Protein_id', 'Pfam'])
pfam_df = pfam_df.groupby("Protein_id")["Pfam"].apply(lambda x: ','.join(x)).reset_index()

ec_number_df = pd.read_csv('EC_number.csv', names=['Protein_id', 'EC_number'])
ec_number_df = ec_number_df.groupby("Protein_id")["EC_number"].apply(lambda x: ','.join(x)).reset_index()

nucleic_acid_df = pd.read_csv('Nucleic_acid_sequence.csv')
nucleic_acid_df.rename(columns={'gene':'Gene','sequence':'Nucleic_acid_sequence'}, inplace=True)

amino_acid_sequence_df = pd.read_csv('Amino_acid_sequence.csv')
amino_acid_sequence_df.rename(columns={'Protein ID':'Protein_id','Sequence':'Amino_acid_sequence'}, inplace=True)


In [18]:
kegg_definition = pd.read_csv('KEGG_definition.csv')
kegg_definition.rename(columns={'Pathway':'KEGG_definition'}, inplace=True)

kegg_df = pd.read_csv('KEGG.csv', names=['KEGG', 'KEGG_definition'])

# 合并kegg_definition，kegg_df
kegg_merge_df = pd.merge(kegg_definition, kegg_df, on='KEGG_definition', how='inner')
kegg_merge_df = kegg_merge_df.fillna('')

kegg_merge_df = kegg_merge_df.groupby("Gene").agg({
    'KEGG_definition': lambda x: ','.join(x),
    'KEGG': lambda x: ','.join(x)
}).reset_index()

display(kegg_merge_df)
kegg_merge_df.to_csv('KEGG_merge.csv', index=False)

,Gene,KEGG_definition,KEGG
0,MYCTH_100519,"Inositol phosphate metabolism,Metabolic pathwa...","mtm00562,mtm01100,mtm04070"
1,MYCTH_100654,"Autophagy - yeast,Protein processing in endopl...","mtm04138,mtm04141"
2,MYCTH_101224,"Fatty acid degradation,Tryptophan metabolism","mtm00071,mtm00380"
3,MYCTH_101411,Aminoacyl-tRNA biosynthesis,mtm00970
4,MYCTH_101588,"Inositol phosphate metabolism,Metabolic pathwa...","mtm00562,mtm01100,mtm04070,mtm04138"
...,...,...,...
2091,MYCTH_t95,Aminoacyl-tRNA biosynthesis,mtm00970
2092,MYCTH_t96,Aminoacyl-tRNA biosynthesis,mtm00970
2093,MYCTH_t97,Aminoacyl-tRNA biosynthesis,mtm00970
2094,MYCTH_t98,Aminoacyl-tRNA biosynthesis,mtm00970


In [13]:
# Go.csv存在一对多情况
go_df = pd.read_csv('Go.csv')
go_df.rename(columns={'GOID':'GO', 'DEFINITION':'GO_definition', 'ONTOLOGY':'GO_type', 'TERM':'ONTOLOGY'}, inplace=True)
go_df = go_df.fillna('')

go_df = go_df.groupby('Gene').agg({
    'GO': lambda x: ','.join(x),
    'GO_definition': lambda x: '/'.join(x),
    'ONTOLOGY': lambda x: ','.join(x),
    'GO_type': lambda x: ','.join(x)
}).reset_index()

go_df

In [ ]:
df = pd.merge(location_df, ncbi_gene_id_df, on='Gene', how='outer')
df = pd.merge(df, protein_id_df, on='Gene', how='outer')
df = pd.merge(df, uniprot_id_df, on='Gene', how='outer')
df = pd.merge(df, kunm_df, on='Protein_id', how='outer')
df = pd.merge(df, description_df, on='Protein_id', how='outer')
df = pd.merge(df, pfam_df, on='Protein_id', how='outer')
df = pd.merge(df, ec_number_df, on='Protein_id', how='outer')
df = pd.merge(df, nucleic_acid_df, on='Gene', how='outer')
df = pd.merge(df, amino_acid_sequence_df, on='Protein_id', how='outer')
df = pd.merge(df, go_df, on='Gene', how='outer')
df = pd.merge(df, kegg_merge_df, on='Gene', how='outer')
df = df.fillna('N/A')

# 将df的Gene这一列按照升序排列，扩展到其他列
df = df.sort_values(by='Gene')
df = df.reset_index(drop=True)
display(df)

df.to_csv('gene_information.csv', index=False)

In [15]:
df_change = pd.read_csv('gene_information.csv')

df_dict = df_change.fillna('null').to_dict(orient='records')

In [ ]:
output_dict = {}

for item in df_dict:
    Gene = 'KEY:' + item['Gene']
    output_dict[Gene] = {
        'Gene': item['Gene'],
        'NCBI_gene_id': int(item['NCBI_gene_id']) if item['NCBI_gene_id'] != 'null' else None,
        'Protein_id': item['Protein_id'],
        'Uniport_id': item['Uniport_id'],
        'Knum': str(item['Knum']),
        'Description': item['Description'],
        'PFAM': item['Pfam'],
        'Chr': item['Chr'],
        'Start': int(item['Start']) if item['Start'] != 'null' else None,
        'End': int(item['End']) if item['End'] != 'null' else None,
        'Strand': item['Strand'],
        'Length': int(item['Length']) if item['Length'] != 'null' else None,
        'Nucleic_acid_sequence': item['Nucleic_acid_sequence'],
        'Amino_acid_sequence': item['Amino_acid_sequence'],
        'GO_information': {
            'GO': item['GO'],
            'GO_definition': item['GO_definition'],
            'ONTOLOGY': item['ONTOLOGY'],
            'GO_type': item['GO_type'],
        },
        'KEGG_information': {
            'KEGG': str(item['KEGG']),
            'KEGG_definition': str(item['KEGG_definition']),
        },
        'EC_number': str(item['EC_number']),
    }

In [ ]:
for key, value in output_dict.items():
    GO = value['GO_information']['GO'].split(',')
    GO_definition = value['GO_information']['GO_definition'].split('/')
    ONTOLOGY = value['GO_information']['ONTOLOGY'].split(',')
    GO_type = value['GO_information']['GO_type'].split(',')
    GO_information = []
    for i in range(len(GO)):
        GO_information.append({
            'GO': GO[i],
            'GO_definition': GO_definition[i],
            'ONTOLOGY': ONTOLOGY[i],
            'GO_type': GO_type[i]
        })
    output_dict[key]['GO_information'] = GO_information

for key, value in output_dict.items():
    KEGG = value['KEGG_information']['KEGG'].split(',')
    KEGG_definition = value['KEGG_information']['KEGG_definition'].split(',')
    KEGG_information = []
    for i in range(len(KEGG)):
        KEGG_information.append({
            'KEGG': KEGG[i],
            'KEGG_definition': KEGG_definition[i],
        })
    output_dict[key]['KEGG_information'] = KEGG_information

# output_dict

In [19]:
with open('gene_information.json', 'w') as f:
    json.dump(output_dict, f, indent=4)